In [9]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # same store the UI uses
mlflow.set_experiment("metro-flow-forecasting")

<Experiment: artifact_location='/mnt/NewData/vs code/Metro-Passengers/mlruns/1', creation_time=1785509289496, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785509289496, lifecycle_stage='active', name='metro-flow-forecasting', tags={}, trace_location=None, workspace='default'>

In [10]:
import pandas as pd
import numpy as np

flow = pd.read_csv("data/processed/flow_features.csv", parse_dates=["interval"])

# the feature columns the model may use (NOT interval, outflow, split, or inflow itself)
FEATURES = ["stationID", "hour", "minute", "dayofweek", "is_weekend",
            "inflow_lag_1", "inflow_lag_2", "inflow_lag_3", "inflow_lag_4",
            "inflow_roll_mean_4"]
TARGET = "inflow"

# split by the split column (NO shuffling — it's already time-ordered)
train = flow[flow["split"] == "train"]
val   = flow[flow["split"] == "val"]
test  = flow[flow["split"] == "test"]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (109270, 10) Val: (12170, 10) Test: (30962, 10)


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{label:20s}  MAE={mae:7.2f}  RMSE={rmse:7.2f}")
    return {"model": label, "mae": mae, "rmse": rmse}

In [12]:
# build the lookup table from TRAINING data only
ha_table = (
    train.groupby(["stationID", "hour", "is_weekend"])["inflow"]
         .mean()
         .rename("ha_pred")
         .reset_index()
)

# a fallback for combos never seen in training
global_mean = train["inflow"].mean()

def historical_average_predict(df):
    merged = df.merge(ha_table, on=["stationID", "hour", "is_weekend"], how="left")
    return merged["ha_pred"].fillna(global_mean).values

# predict on validation and test
ha_val_pred  = historical_average_predict(val)
ha_test_pred = historical_average_predict(test)

results = []
results.append(evaluate(y_val,  ha_val_pred,  "HistoricalAvg (val)"))
results.append(evaluate(y_test, ha_test_pred, "HistoricalAvg (test)"))

HistoricalAvg (val)   MAE=  29.34  RMSE=  60.27
HistoricalAvg (test)  MAE=  37.55  RMSE=  72.94


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

# which columns get which treatment
categorical = ["stationID"]
numeric = ["hour", "minute", "dayofweek", "is_weekend",
           "inflow_lag_1", "inflow_lag_2", "inflow_lag_3", "inflow_lag_4",
           "inflow_roll_mean_4"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
])

ridge_pipe = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0)),
])

# fit on TRAIN only
ridge_pipe.fit(X_train, y_train)

# predict on val and test
ridge_val_pred  = ridge_pipe.predict(X_val)
ridge_test_pred = ridge_pipe.predict(X_test)

results.append(evaluate(y_val,  ridge_val_pred,  "Ridge (val)"))
results.append(evaluate(y_test, ridge_test_pred, "Ridge (test)"))

Ridge (val)           MAE=  29.29  RMSE=  45.48
Ridge (test)          MAE=  39.29  RMSE=  74.52


In [14]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

# XGBoost can use the raw feature matrix directly (station as numeric is fine for trees)
xgb.fit(X_train, y_train)

xgb_val_pred  = xgb.predict(X_val)
xgb_test_pred = xgb.predict(X_test)

results.append(evaluate(y_val,  xgb_val_pred,  "XGBoost (val)"))
results.append(evaluate(y_test, xgb_test_pred, "XGBoost (test)"))

XGBoost (val)         MAE=  22.42  RMSE=  37.23
XGBoost (test)        MAE=  25.75  RMSE=  51.80


In [15]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("metro-flow-forecasting")

def log_run(name, model, params, val_pred, test_pred, is_xgb=False):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        # metrics
        mlflow.log_metric("val_mae",  mean_absolute_error(y_val, val_pred))
        mlflow.log_metric("val_rmse", np.sqrt(mean_squared_error(y_val, val_pred)))
        mlflow.log_metric("test_mae", mean_absolute_error(y_test, test_pred))
        mlflow.log_metric("test_rmse", np.sqrt(mean_squared_error(y_test, test_pred)))
        # the model artifact
        if is_xgb:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")
        print(f"logged: {name}")

# Ridge
log_run("ridge", ridge_pipe,
        {"model_type": "Ridge", "alpha": 1.0},
        ridge_val_pred, ridge_test_pred)

# XGBoost
log_run("xgboost", xgb,
        {"model_type": "XGBoost", "n_estimators": 300, "learning_rate": 0.1, "max_depth": 6},
        xgb_val_pred, xgb_test_pred, is_xgb=True)

logged: ridge
logged: xgboost


In [16]:
with mlflow.start_run(run_name="historical_average"):
    mlflow.log_param("model_type", "HistoricalAverage")
    mlflow.log_param("grouping", "stationID_hour_is_weekend")
    mlflow.log_metric("val_mae",  mean_absolute_error(y_val, ha_val_pred))
    mlflow.log_metric("val_rmse", np.sqrt(mean_squared_error(y_val, ha_val_pred)))
    mlflow.log_metric("test_mae", mean_absolute_error(y_test, ha_test_pred))
    mlflow.log_metric("test_rmse", np.sqrt(mean_squared_error(y_test, ha_test_pred)))
    print("logged: historical_average")

logged: historical_average


In [19]:
import os
# ensure we're at the repo root no matter where the kernel started
if os.path.basename(os.getcwd()) != "metro-flow-forecasting":
    os.chdir("metro-flow-forecasting")
print("Working dir:", os.getcwd())

Working dir: /mnt/NewData/vs code/Metro-Passengers/metro-flow-forecasting


In [20]:
import os
print("parent data:", os.path.isdir("data"))
print("repo data:", os.path.isdir("metro-flow-forecasting/data"))

parent data: True
repo data: False


In [21]:
import os
os.makedirs("artifacts", exist_ok=True)
import joblib
artifact = {
    "model": xgb, "features": FEATURES, "model_type": "XGBoost",
    "val_mae": float(mean_absolute_error(y_val, xgb_val_pred)),
    "test_mae": float(mean_absolute_error(y_test, xgb_test_pred)),
}
joblib.dump(artifact, "artifacts/metro_xgb_model.joblib")
print("Saved:", os.path.getsize("artifacts/metro_xgb_model.joblib"), "bytes")

Saved: 1272602 bytes


In [22]:
import joblib
import numpy as np

# load from disk as if in a fresh environment
loaded = joblib.load("artifacts/metro_xgb_model.joblib")
loaded_model = loaded["model"]
loaded_features = loaded["features"]

# predict on test using the reloaded model — no retraining
reload_pred = loaded_model.predict(X_test[loaded_features])

# must exactly match the original predictions
matches = np.allclose(reload_pred, xgb_test_pred)
print("Reload predictions match original:", matches)
print("Sample — predicted:", round(float(reload_pred[0]), 1), "| actual:", int(y_test.iloc[0]))
print("Reloaded model val_mae:", round(loaded["val_mae"], 2))

Reload predictions match original: True
Sample — predicted: 3.3 | actual: 0
Reloaded model val_mae: 22.42
